# District CSV Cleaning

## Setup

Load and inspect the district profile CSV before making any changes.

In [1]:
import pandas as pd
import re

district = pd.read_csv(
    "db-unza26-csc4792-zimba_town_council_district_profile.csv",
    sep="|",
)

print("Shape:", district.shape)
print("Head:")
print(district.head())
print("Info:")
district.info()
print("Describe:")
print(district.describe(include="all"))

Shape: (4, 14)
Head:
      record_id         level       name  population  population_year  \
0  ZTC-DIST-001      district      Zimba     66725.0           2010.0   
1  ZTC-DIST-002      district      Zimba     98533.0           2018.0   
2  ZTC-DIST-003      district      Zimba    108316.0           2022.0   
3  ZTC-DIST-004  constituency  Mapatizya         NaN              NaN   

   population_male  population_female  households  area_km2  province  \
0          32186.0            34539.0     13284.0    5245.0  Southern   
1              NaN                NaN         NaN    5245.0  Southern   
2              NaN                NaN         NaN    5245.0  Southern   
3              NaN                NaN         NaN       NaN  Southern   

               neighboring_districts  \
0  Kalomo;Kazungula;Choma;Sinazongwe   
1  Kalomo;Kazungula;Choma;Sinazongwe   
2  Kalomo;Kazungula;Choma;Sinazongwe   
3                                NaN   

                                              

## 1.2 Detect → Judge → Act: Duplicates

This check applies only to the district profile. A duplicate district-profile entity is identified by the same `name` and `level`. Rows with different `name + level` values are retained.

In [2]:
# Detect: check exact duplicates and near-duplicates based on name + level.
exact_duplicates = district.duplicated(keep=False)
near_duplicates = district.duplicated(subset=["name", "level"], keep=False)

print("District - Detect")
print(f"Exact duplicate rows: {exact_duplicates.sum()}")
print(f"Rows involved in repeated name + level values: {near_duplicates.sum()}")

if near_duplicates.any():
    print("Repeated name + level values:")
    print(
        district.loc[near_duplicates, ["name", "level"]]
        .sort_values(["name", "level"])
    )

# Judge: the same name + level identifies the same district-profile entity.
# Act: keep the first occurrence and remove later occurrences.
before = len(district)
district = district.drop_duplicates(
    subset=["name", "level"],
    keep="first",
).copy()

removed = before - len(district)
print("District - Judge: duplicate key is name + level")
print(f"District - Act: removed {removed} duplicate row(s)")
print(f"Rows after: {len(district)}")

District - Detect
Exact duplicate rows: 0
Rows involved in repeated name + level values: 3
Repeated name + level values:
    name     level
0  Zimba  district
1  Zimba  district
2  Zimba  district
District - Judge: duplicate key is name + level
District - Act: removed 2 duplicate row(s)
Rows after: 2


## 1.3 Detect → Judge → Act: Missing Values

### Detect

Run `isnull().sum()` on the district DataFrame and identify columns with gaps.

### Judge and Act

| Column | Judgement | Action |
| --- | --- | --- |
| `population` | Target-like demographic measure | Keep the blank for the constituency row because the council does not publish a constituency population. Do not invent a value or drop the row. |
| `population_year` | Supports interpretation of `population` | Keep blank when no population is published. |
| `population_male` / `population_female` | Supporting demographic breakdowns | Keep blank where the source does not provide a breakdown. |
| `households` | Supporting demographic measure | Keep blank where the source does not provide a household count. |
| `area_km2` | Target-like geographic measure | Keep the blank for the constituency row because no constituency area is published. |
| `neighboring_districts` | Supporting district feature | Keep blank for the constituency row because neighboring districts are recorded at district level. |
| `secondary_source_url` | Source evidence | Keep blank when no secondary source was needed. |

No rows are dropped for missing values: the blanks are source-supported and the constituency record remains useful.

In [3]:
# Detect: count missing values in every district column.
missing_values = district.isnull().sum()
print("District - Missing values detected")
print(missing_values)

print("\nColumns with gaps:")
print(missing_values[missing_values > 0])

# Judge and Act: preserve source-supported blanks rather than inventing values.
# No rows are dropped because the missing values are legitimate for the record level.
district_clean = district.copy()

print("\nDistrict - Judge: missing values are source-supported by record level")
print("District - Act: dropped 0 rows and left legitimate blanks unchanged")
print(f"Rows after missing-value handling: {len(district_clean)}")

District - Missing values detected
record_id                0
level                    0
name                     0
population               1
population_year          1
population_male          1
population_female        1
households               1
area_km2                 1
province                 0
neighboring_districts    1
notes                    0
source_url               0
secondary_source_url     2
dtype: int64

Columns with gaps:
population               1
population_year          1
population_male          1
population_female        1
households               1
area_km2                 1
neighboring_districts    1
secondary_source_url     2
dtype: int64

District - Judge: missing values are source-supported by record level
District - Act: dropped 0 rows and left legitimate blanks unchanged
Rows after missing-value handling: 2


## 1.4 Detect → Judge → Act: Outliers

### Detect

For this district file, `population` and `area_km2` are the relevant numeric measures. `funding_amount_zmw` belongs to the CDF projects dataset and is not present here, so it is not applicable.

The IQR rule is:

- Lower bound = Q1 - 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

### Judge and Act

An IQR flag is only a prompt to inspect the row's `source_url`; it is not proof of an error. Genuine large populations or areas must be retained. No value is corrected or removed without source confirmation. Any confirmed issue would be corrected in a separate, documented action.

In [6]:
# Detect: compute IQR bounds for numeric columns relevant to the district file.
numeric_columns = ["funding_amount_zmw", "population", "area_km2"]

print("District - Outlier detection")
for column in numeric_columns:
    if column not in district_clean.columns:
        print(f"{column}: not applicable (column is not in the district file)")
        continue

    values = district_clean[column].dropna()
    if values.empty:
        print(f"{column}: no non-missing values to assess")
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    flagged = district_clean[
        district_clean[column].notna()
        & (
            (district_clean[column] < lower_bound)
            | (district_clean[column] > upper_bound)
        )
    ]

    print(f"\n{column}")
    print(f"Q1: {q1}")
    print(f"Q3: {q3}")
    print(f"IQR: {iqr}")
    print(f"Lower bound: {lower_bound}")
    print(f"Upper bound: {upper_bound}")
    print(f"Flagged rows: {len(flagged)}")

    if not flagged.empty:
        print("Review these source URLs before taking action:")
        print(flagged[["record_id", "name", "level", column, "source_url"]])

# Judge and Act: IQR flags are not automatically errors.
# Keep all values because no source-confirmed data-entry or scraping error was found.
print("\nDistrict - Judge: large values are not automatically wrong")
print("District - Act: removed 0 rows and corrected 0 values")

District - Outlier detection
funding_amount_zmw: not applicable (column is not in the district file)

population
Q1: 66725.0
Q3: 66725.0
IQR: 0.0
Lower bound: 66725.0
Upper bound: 66725.0
Flagged rows: 0

area_km2
Q1: 5245.0
Q3: 5245.0
IQR: 0.0
Lower bound: 5245.0
Upper bound: 5245.0
Flagged rows: 0

District - Judge: large values are not automatically wrong
District - Act: removed 0 rows and corrected 0 values


## 1.5 Text Cleaning: District Notes

The district file has no `description` column. Its free-text field is `notes`, so the same Lecture 4 pipeline is applied to a new `notes_clean` column while preserving the original `notes` column.

The order is: HTML tag removal (Step 0) → case folding → punctuation removal → tokenization → stopword removal. Missing notes are kept as blank text, not guessed.

In [5]:
import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

stop_words = set(stopwords.words("english"))


def clean_text(text):
    if pd.isna(text):
        return ""

    text = re.sub(r"<[^>]+>", "", str(text))  # Step 0: HTML tags
    text = text.lower()  # Case folding
    text = "".join(c for c in text if c not in string.punctuation)  # Punctuation
    tokens = word_tokenize(text)  # Tokenization
    tokens = [token for token in tokens if token not in stop_words]  # Stopwords
    return " ".join(tokens)


# Keep the original notes column and create a cleaned copy.
district_clean["notes_clean"] = district_clean["notes"].apply(clean_text)

print("District - Text cleaning complete")
print("Original notes column preserved: notes")
print("New cleaned column: notes_clean")
print(district_clean[["notes", "notes_clean"]])

District - Text cleaning complete
Original notes column preserved: notes
New cleaned column: notes_clean
                                               notes  \
0  2010 Census by the Central Statistical Office ...   
3  Named as a constituency in Zimba Town Council'...   

                                         notes_clean  
0  2010 census central statistical office cso ann...  
3  named constituency zimba town councils cdf fin...  


## 1.6 Standardise Types and Categories

This section applies only to columns present in the district CSV. `date_reported` and `funding_amount_zmw` belong to other datasets and are not present here.

- Convert `population` and `area_km2` to numeric values using `errors="coerce"`.
- Standardize categorical `level` values with lowercase and whitespace removal.
- No filtering is performed here; any future filter must use `==` for comparison, never `=`.

In [7]:
# Standardise numeric columns available in the district file.
numeric_columns = ["population", "area_km2"]
for column in numeric_columns:
    district_clean[column] = pd.to_numeric(
        district_clean[column],
        errors="coerce",
    )

# These columns belong to other datasets and are not present in district_clean.
for column in ["date_reported", "funding_amount_zmw"]:
    if column not in district_clean.columns:
        print(f"{column}: not applicable to the district file")

# Standardise categorical columns available in the district file.
for column in ["status", "sector", "level"]:
    if column in district_clean.columns:
        district_clean[column] = (
            district_clean[column]
            .astype("string")
            .str.strip()
            .str.lower()
        )
        print(f"Standardised category: {column}")

# Any future row filter must use == for comparison, never =.
print("Population dtype:", district_clean["population"].dtype)
print("Area dtype:", district_clean["area_km2"].dtype)
print("Levels:", district_clean["level"].dropna().unique().tolist())

date_reported: not applicable to the district file
funding_amount_zmw: not applicable to the district file
Standardised category: level
Population dtype: float64
Area dtype: float64
Levels: ['district', 'constituency']


## 1.7 Export the Clean District File

Export the district-only cleaned DataFrame using the project's original filename, pipe separator, and no index column. The exported file is reopened to verify the round trip.

In [8]:
from pathlib import Path

output_path = Path("db-unza26-csc4792-zimba_town_council_district_profile.csv")
district_clean.to_csv(output_path, sep="|", index=False)

# Reopen the exported file to verify the separator and index handling.
district_exported = pd.read_csv(output_path, sep="|")

print(f"Exported: {output_path}")
print("Shape:", district_exported.shape)
print("Separator check: pipe-delimited file reopened successfully")
print("Index check: unnamed index column present:", "Unnamed: 0" in district_exported.columns)
print("Columns:")
print(district_exported.columns.tolist())

Exported: db-unza26-csc4792-zimba_town_council_district_profile.csv
Shape: (2, 15)
Separator check: pipe-delimited file reopened successfully
Index check: unnamed index column present: False
Columns:
['record_id', 'level', 'name', 'population', 'population_year', 'population_male', 'population_female', 'households', 'area_km2', 'province', 'neighboring_districts', 'notes', 'source_url', 'secondary_source_url', 'notes_clean']


## 1.8 Before You Hand Off

The district file started with 4 rows and 14 columns. Two duplicate rows were removed using the `name` + `level` key, leaving 2 cleaned rows; the exported file has 15 columns because `notes_clean` was added while preserving the original `notes`. Missing values remain in `population`, `population_year`, `population_male`, `population_female`, `households`, `area_km2`, `neighboring_districts`, and `secondary_source_url` because they are source-supported, especially for the constituency record, or document that no secondary source was needed. No population or area outliers were flagged by the IQR check, so no values were corrected or removed.

In [9]:
# Record final row and column counts for the district handoff.
final_row_count = len(district_exported)
final_column_count = len(district_exported.columns)

print("Final district row count:", final_row_count)
print("Final district column count:", final_column_count)
print("Duplicate rows removed:", 2)
print("Outliers corrected or removed:", 0)

Final district row count: 2
Final district column count: 15
Duplicate rows removed: 2
Outliers corrected or removed: 0
